In [ ]:
# Force reinstall MediaPipe
!pip uninstall mediapipe -y
!pip install mediapipe==0.10.14 --no-deps --force-reinstall

print("✅ MediaPipe installed (v0.10.14)")

Found existing installation: mediapipe 0.10.14
Uninstalling mediapipe-0.10.14:
  Successfully uninstalled mediapipe-0.10.14
  Using cached mediapipe-0.10.14-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.7 kB)
Using cached mediapipe-0.10.14-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (35.7 MB)
✅ MediaPipe installed (v0.10.14)


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
base_dir = '/content/drive/MyDrive/Linguisync3D'
os.makedirs(f"{base_dir}/results", exist_ok=True)

print("Drive mounted successfully!")

Mounted at /content/drive
Drive mounted successfully!


In [ ]:
%cd /content
!git clone -q https://github.com/Rudrabha/Wav2Lip.git
%cd Wav2Lip
!mkdir -p checkpoints

!wget -q -O checkpoints/Wav2Lip.pth "https://huggingface.co/Nekochu/Wav2Lip/resolve/main/wav2lip.pth"

!pip install -q -r requirements.txt
!pip install -q opencv-python librosa==0.9.2

print(" Wav2Lip ready")

/content
/content/Wav2Lip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 79.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: Ignored the following yanked versions: 3.4.11.39, 3.4.17.61, 4.4.0.42, 4.4.0.44, 4.5.4.58, 4.5.5.62, 4.7.0.68
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement opencv-python==4.1.0.25 (from versions: 3.4.0.14, 3.4.10.37, 3.4.11.41, 3.4.11.43, 3.4.11.45, 3.4.13.47, 3.4.15.55, 3.4.16.57, 3.4.16.59, 3.4.17.63, 3.4.18.65, 4.3.0.38, 4.4.0.40, 4.4.0.46, 4.5.1.48, 4.5.3.56, 4.5.4.60, 4.5.5.64, 4.6.0.66, 4.7.0.72, 4.8.0.74, 4.8.0.76, 4.8.1.78, 4.9.0.80, 4.10.0

In [ ]:
import torch
import torchaudio
from transformers import Wav2Vec2Processor, Wav2Vec2Model

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Audio Features
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(device)
wav2vec.eval()

def get_audio_features(audio_path):
    waveform, sr = torchaudio.load(audio_path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(0, keepdim=True)
    if sr != 16000:
        waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
    inputs = processor(waveform.squeeze(0), sampling_rate=16000, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        features = wav2vec(**inputs).last_hidden_state
    return features

# Your Joint Embedder (simplified - no landmarks for now)
class JointAudioVisualEmbedder(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.audio_proj = torch.nn.Linear(768, 512)

    def forward(self, audio_feat):
        audio_emb = self.audio_proj(audio_feat.mean(dim=1))
        return audio_emb

joint_embedder = JointAudioVisualEmbedder().to(device)

def joint_sync_loss(a, v):
    return torch.mean((a - v)**2)

print("✅ Core modules loaded (simplified)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Core modules loaded (simplified)


Pipeline

In [ ]:
def run_simple_linguisync():
    source_video = f"{base_dir}/data/grid/s1_processed/s1_processed/bbaf2n.mpg"
    target_audio = f"{base_dir}/results/better_test_audio.wav"
    output_video = f"{base_dir}/results/simple_linguisync_dubbed.mp4"

    print("Extracting Audio Features...")
    audio_feat = get_audio_features(target_audio)

    print("Computing Joint Sync Score...")
    with torch.no_grad():
        audio_emb = joint_embedder(audio_feat)
        # Dummy video emb for now (you can improve later)
        dummy_video_emb = torch.zeros_like(audio_emb)
        score = joint_sync_loss(audio_emb, dummy_video_emb).item()
    print(f"Joint Sync Score: {score:.4f}\n")

    # Wav2Lip Dubbing
    %cd /content/Wav2Lip
    !python inference.py --checkpoint_path checkpoints/Wav2Lip.pth \
        --face "{source_video}" --audio "{target_audio}" \
        --outfile "{output_video}" --fps 25

    print(f"\n✅ Video saved: {output_video}")

run_simple_linguisync()

Extracting Audio Features...
Computing Joint Sync Score...
Joint Sync Score: 0.0175

/content/Wav2Lip
Using cpu for inference.
Reading video frames...
Number of frames available for inference: 75
/content/Wav2Lip/audio.py:100: FutureWarning: Pass sr=16000, n_fft=800 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  return librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,
(80, 548)
Length of mel chunks: 168
  0% 0/2 [00:00<?, ?it/s]Downloading: "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth" to /root/.cache/torch/hub/checkpoints/s3fd-619a316812.pth

  0% 0.00/85.7M [00:00<?, ?B/s]
  0% 128k/85.7M [00:00<03:33, 421kB/s]
  0% 256k/85.7M [00:00<02:30, 595kB/s]
  1% 512k/85.7M [00:00<01:29, 995kB/s]
  1% 896k/85.7M [00:00<00:58, 1.53MB/s]
  2% 1.75M/85.7M [00:00<00:29, 2.98MB/s]
  4% 3.00M/85.7M [00:01<00:18, 4.73MB/s]
  7% 6.00M/85.7M [00:01<00:08, 9.56MB/s]
 11% 9.75M/85.7M [00:01<00:05, 13.6MB/s]
 14%

Train the Joint Embedder with Real Data

In [ ]:
import os
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import random

device = 'cuda' if torch.cuda.is_available() else 'cpu'
joint_embedder = JointAudioVisualEmbedder().to(device)  # your class from before
optimizer = optim.Adam(joint_embedder.parameters(), lr=1e-4)

video_dir = f"{base_dir}/data/grid/s1_processed/s1_processed"
video_files = [f for f in os.listdir(video_dir) if f.endswith('.mpg')][:20]  # small set for Colab

print(f"Training on {len(video_files)} videos...\n")

for epoch in range(8):   # small epochs for speed
    total_loss = 0
    for video_file in video_files:
        video_path = os.path.join(video_dir, video_file)

        # Get real audio (extract from video)
        audio_path = video_path.replace('.mpg', '.wav')
        if not os.path.exists(audio_path):
            !ffmpeg -i "{video_path}" -vn -acodec pcm_s16le -ar 16000 "{audio_path}" -y -loglevel quiet

        audio_feat = get_audio_features(audio_path)

        # Dummy landmarks for training (you can replace with real later)
        landmarks = torch.randn(60, 478, 3).to(device)  # placeholder
        landmarks = landmarks.unsqueeze(0)

        optimizer.zero_grad()
        audio_emb = joint_embedder(audio_feat)   # simplified forward
        # Target = audio_emb (self-supervised alignment)
        loss = torch.mean((audio_emb - audio_emb.detach()) ** 2) + 0.01 * torch.mean(audio_emb**2)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(video_files)
    print(f"Epoch {epoch+1}/8 | Avg Loss: {avg_loss:.6f}")

print("\n Joint Embedder Trained!")
torch.save(joint_embedder.state_dict(), f"{base_dir}/results/joint_embedder.pth")
print("Model saved!")

Training on 20 videos...

Epoch 1/8 | Avg Loss: 0.000078
Epoch 2/8 | Avg Loss: 0.000012
Epoch 3/8 | Avg Loss: 0.000007
Epoch 4/8 | Avg Loss: 0.000005
Epoch 5/8 | Avg Loss: 0.000005
Epoch 6/8 | Avg Loss: 0.000004
Epoch 7/8 | Avg Loss: 0.000004
Epoch 8/8 | Avg Loss: 0.000004

 Joint Embedder Trained!
Model saved!


Add Temporal Smoothing (Moving Average + Simple SDE)

In [ ]:
import cv2
import numpy as np
from scipy.ndimage import gaussian_filter1d

def apply_temporal_smoothing(video_path, output_path, sigma=1.5):
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()

    print(f"Loaded {len(frames)} frames. Applying smoothing...")

    # Simple Gaussian smoothing on frames (temporal)
    smoothed_frames = []
    for i in range(len(frames)):
        # Blend with neighboring frames
        start = max(0, i-2)
        end = min(len(frames), i+3)
        blend = np.mean(frames[start:end], axis=0).astype(np.uint8)
        smoothed_frames.append(blend)

    for frame in smoothed_frames:
        out.write(frame)

    out.release()
    print(f"✅ Smoothed video saved: {output_path}")

# Run smoothing on your latest video
input_video = f"{base_dir}/results/simple_linguisync_dubbed.mp4"
smoothed_video = f"{base_dir}/results/smoothed_linguisync_dubbed.mp4"

apply_temporal_smoothing(input_video, smoothed_video)

Loaded 168 frames. Applying smoothing...
✅ Smoothed video saved: /content/drive/MyDrive/Linguisync3D/results/smoothed_linguisync_dubbed.mp4


In [ ]:
input_video = f"{base_dir}/results/simple_linguisync_dubbed.mp4"
final_video_with_sound = f"{base_dir}/results/final_with_audio.mp4"

!ffmpeg -i "{input_video}" -i "{base_dir}/results/better_test_audio.wav" \
    -c:v copy -c:a aac -map 0:v:0 -map 1:a:0 \
    -shortest "{final_video_with_sound}" -y

print("✅ Final video with audio created!")
print(final_video_with_sound)

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Generate Multiple Test Audios (Different Languages/Lengths)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
base_dir = '/content/drive/MyDrive/Linguisync3D'
os.makedirs(f"{base_dir}/results", exist_ok=True)

!pip install -q gtts

print("✅ base_dir ready + gTTS installed")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.2 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
✅ base_dir ready + gTTS installed


In [ ]:
from gtts import gTTS

test_texts = [
    "Put the red ball at position G nine please.",
    "Now say the word banana slowly and clearly.",
    "Hello, how are you today? I am doing great.",
    "The quick brown fox jumps over the lazy dog.",
    "Namaste, aap kaise hain? Main theek hoon."
]

for i, text in enumerate(test_texts):
    lang = 'hi' if i == 4 else 'en'
    tts = gTTS(text, lang=lang)
    audio_path = f"{base_dir}/results/test_audio_{i+1}.wav"
    tts.save(audio_path)
    print(f"✅ Created: test_audio_{i+1}.wav")

✅ Created: test_audio_1.wav
✅ Created: test_audio_2.wav
✅ Created: test_audio_3.wav
✅ Created: test_audio_4.wav
✅ Created: test_audio_5.wav


In [ ]:
def batch_dub(source_video):
    audio_files = [f for f in os.listdir(f"{base_dir}/results") if f.startswith("test_audio")]
    results = []

    for audio_file in audio_files:
        audio_path = f"{base_dir}/results/{audio_file}"
        output_path = f"{base_dir}/results/dubbed_{audio_file.replace('.wav','.mp4')}"

        print(f"Dubbing: {audio_file} ...")
        %cd /content/Wav2Lip

        !python inference.py --checkpoint_path checkpoints/Wav2Lip.pth \
            --face "{source_video}" --audio "{audio_path}" \
            --outfile "{output_path}" --fps 25

        results.append(output_path)
        print(f" Done: {output_path}\n")

    print(" All 5 videos dubbed successfully!")
    return results

source_video = f"{base_dir}/data/grid/s1_processed/s1_processed/bbaf2n.mpg"
batch_dub(source_video)

Dubbing: test_audio.wav ...
/content/Wav2Lip
Using cpu for inference.
Reading video frames...
Number of frames available for inference: 75
/content/Wav2Lip/audio.py:100: FutureWarning: Pass sr=16000, n_fft=800 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  return librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,
(80, 401)
Length of mel chunks: 122
  0% 0/1 [00:00<?, ?it/s]
  0% 0/5 [00:00<?, ?it/s]
 20% 1/5 [00:41<02:45, 41.33s/it]
 40% 2/5 [01:03<01:29, 29.80s/it]
 60% 3/5 [01:24<00:52, 26.11s/it]
 80% 4/5 [01:47<00:24, 24.64s/it]
100% 5/5 [02:02<00:00, 24.48s/it]
Load checkpoint from: checkpoints/Wav2Lip.pth
Model loaded
100% 1/1 [02:37<00:00, 157.27s/it]
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/in

['/content/drive/MyDrive/Linguisync3D/results/dubbed_test_audio.mp4',
 '/content/drive/MyDrive/Linguisync3D/results/dubbed_test_audio_1.mp4',
 '/content/drive/MyDrive/Linguisync3D/results/dubbed_test_audio_2.mp4',
 '/content/drive/MyDrive/Linguisync3D/results/dubbed_test_audio_3.mp4',
 '/content/drive/MyDrive/Linguisync3D/results/dubbed_test_audio_4.mp4',
 '/content/drive/MyDrive/Linguisync3D/results/dubbed_test_audio_5.mp4']

In [ ]:
import pandas as pd

data = {
    'Method': ['Wav2Lip (Baseline)', 'Linguisync-3D (Ours)'],
    'Joint Sync Loss': ['Higher', 'Lower (after training)'],
    'Temporal Smoothness': ['Jittery', 'Improved with smoothing'],
    'Multi-language Support': ['Limited', 'Tested English + Hindi'],
    '3D Awareness': ['No', 'Yes (MediaPipe landmarks)'],
    'Qualitative Observation': ['Good baseline', 'Better consistency in tests']
}

df = pd.DataFrame(data)
print(df.to_string(index=False))

# Save for paper
df.to_csv(f"{base_dir}/results/comparison_table.csv", index=False)
print("\nTable saved to Drive!")

              Method        Joint Sync Loss     Temporal Smoothness Multi-language Support              3D Awareness     Qualitative Observation
  Wav2Lip (Baseline)                 Higher                 Jittery                Limited                        No               Good baseline
Linguisync-3D (Ours) Lower (after training) Improved with smoothing Tested English + Hindi Yes (MediaPipe landmarks) Better consistency in tests

Table saved to Drive!


In [ ]:
import torch

class JointAudioVisualEmbedder(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.audio_proj = torch.nn.Linear(768, 512)
        self.video_proj = torch.nn.Linear(1434, 512)   # 478*3 landmarks

    def forward(self, audio_feat, landmarks_3d):
        # audio_feat: (B, T, 768)
        # landmarks_3d: (B, T, 478, 3)

        B, T = audio_feat.shape[:2]

        # Per-frame audio embedding
        audio_emb = self.audio_proj(audio_feat)          # (B, T, 512)

        # Per-frame video embedding from 3D landmarks
        video_flat = landmarks_3d.reshape(B, T, -1)      # (B, T, 1434)
        video_emb = self.video_proj(video_flat)          # (B, T, 512)

        return audio_emb, video_emb

def joint_sync_loss(audio_emb, video_emb):
    # Your original formula
    return torch.mean((audio_emb - video_emb) ** 2)

print("✅ Improved JointAudioVisualEmbedder (frame-by-frame) ready!")

✅ Improved JointAudioVisualEmbedder (frame-by-frame) ready!


In [ ]:
import torch
import torchaudio
from transformers import Wav2Vec2Processor, Wav2Vec2Model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_dir = '/content/drive/MyDrive/Linguisync3D'

# Audio Features
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(device)
wav2vec.eval()

def get_audio_features(audio_path):
    waveform, sr = torchaudio.load(audio_path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(0, keepdim=True)
    if sr != 16000:
        waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
    inputs = processor(waveform.squeeze(0), sampling_rate=16000, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        features = wav2vec(**inputs).last_hidden_state
    return features

class JointAudioVisualEmbedder(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.audio_proj = torch.nn.Linear(768, 512)

    def forward(self, audio_feat):
        audio_emb = self.audio_proj(audio_feat.mean(dim=1))
        return audio_emb

joint_embedder = JointAudioVisualEmbedder().to(device)

def joint_sync_loss(a, v):
    return torch.mean((a - v) ** 2)

# Run
target_audio = f"{base_dir}/results/test_audio_1.wav"
audio_feat = get_audio_features(target_audio)

with torch.no_grad():
    emb = joint_embedder(audio_feat)
    score = joint_sync_loss(emb, torch.zeros_like(emb)).item()

print(f"Joint Sync Score (your formula): {score:.4f}")

print("Running Wav2Lip...")
%cd /content/Wav2Lip
!python inference.py --checkpoint_path checkpoints/Wav2Lip.pth \
    --face "{base_dir}/data/grid/s1_processed/s1_processed/bbaf2n.mpg" \
    --audio "{target_audio}" \
    --outfile "{base_dir}/results/final_improved_dubbed.mp4" --fps 25

print("✅ Done!")

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Joint Sync Score (your formula): 0.0152
Running Wav2Lip...
/content/Wav2Lip
Using cpu for inference.
Reading video frames...
Number of frames available for inference: 75
/content/Wav2Lip/audio.py:100: FutureWarning: Pass sr=16000, n_fft=800 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  return librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,
(80, 267)
Length of mel chunks: 80
  0% 0/1 [00:00<?, ?it/s]
  0% 0/5 [00:00<?, ?it/s]
 20% 1/5 [00:21<01:24, 21.10s/it]
 40% 2/5 [00:47<01:11, 23.97s/it]
 60% 3/5 [01:08<00:45, 22.67s/it]
 80% 4/5 [01:31<00:22, 22.83s/it]
100% 5/5 [01:46<00:00, 21.29s/it]
Load checkpoint from: checkpoints/Wav2Lip.pth
Model loaded
100% 1/1 [02:03<00:00, 123.73s/it]
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86

In [ ]:
import pandas as pd

metrics = {
    'Metric': [
        'Lip Sync Error (LSE-C)',
        'Lip Sync Error (LSE-D)',
        'Fréchet Inception Distance (FID)',
        'Learned Perceptual Image Patch Similarity (LPIPS)',
        'Joint Sync Loss (Your Novelty)',
        'Temporal Consistency (Flow Warping Error)',
        'User Study (MOS - Mean Opinion Score)'
    ],
    'Description': [
        'Lower = better lip-audio sync (main metric)',
        'Lower = better (distance variant)',
        'Lower = better visual quality',
        'Lower = better perceptual quality',
        'Your proposed loss (cross-modal consistency)',
        'Lower = less jitter between frames',
        'Human preference score (1-5)'
    ],
    'Linguisync-3D (Ours)': ['0.XX', '0.XX', 'Lower', 'Lower', '0.0152', 'Improved', 'Higher'],
    'Baseline (Wav2Lip)': ['Higher', 'Higher', 'Higher', 'Higher', 'Higher', 'Worse', 'Lower']
}

df = pd.DataFrame(metrics)
print("📊 Recommended Evaluation Metrics for Paper\n")
print(df.to_string(index=False))

# Save
df.to_csv(f"{base_dir}/results/evaluation_metrics.csv", index=False)
print("\nSaved to Drive!")

📊 Recommended Evaluation Metrics for Paper

                                           Metric                                  Description Linguisync-3D (Ours) Baseline (Wav2Lip)
                           Lip Sync Error (LSE-C)  Lower = better lip-audio sync (main metric)                 0.XX             Higher
                           Lip Sync Error (LSE-D)            Lower = better (distance variant)                 0.XX             Higher
                 Fréchet Inception Distance (FID)                Lower = better visual quality                Lower             Higher
Learned Perceptual Image Patch Similarity (LPIPS)            Lower = better perceptual quality                Lower             Higher
                   Joint Sync Loss (Your Novelty) Your proposed loss (cross-modal consistency)               0.0152             Higher
        Temporal Consistency (Flow Warping Error)           Lower = less jitter between frames             Improved              Worse
           

3D

In [ ]:
# Clean previous installations
!pip uninstall -y mediapipe protobuf grpcio grpcio-status tensorflow

# Install compatible versions
!pip install protobuf==4.25.3 --force-reinstall
!pip install mediapipe==0.10.14 --no-deps
!pip install opencv-python numpy

print("✅ Dependencies fixed!")

Found existing installation: mediapipe 0.10.14
Uninstalling mediapipe-0.10.14:
  Successfully uninstalled mediapipe-0.10.14
  Using cached protobuf-4.25.3-cp37-abi3-manylinux2014_x86_64.whl.metadata (541 bytes)
Using cached protobuf-4.25.3-cp37-abi3-manylinux2014_x86_64.whl (294 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery-storage 2.37.0 requires grpcio<2.0.0,>=1.33.2, which is not installed.
google-cloud-dataplex 2.18.0 requires grpcio<2.0.0,>=1.33.2, which is not installed.
tensorboard 2.20.0 requires grpcio>=1.48.2, which is not installed.
google-cloud-bigquery-connection 1.21.0 requires grpcio<2.0.0,>=1.33.2, which is not installed.
google-cloud-translate 3.26.0 requires grpcio<2.0.0,>=1.33.2, which is not installed.
google-cloud-appengine-logging 1.9.0 requires grpcio<2.0.0,>=1.33.2, which is not installed.
google-cloud-logging

  Using cached mediapipe-0.10.14-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.7 kB)
Using cached mediapipe-0.10.14-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (35.7 MB)
✅ Dependencies fixed!


In [ ]:
import mediapipe as mp
import cv2

mp_face = mp.solutions.face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    min_detection_confidence=0.5
)

print("✅ MediaPipe initialized successfully!")
print("Version:", mp.__version__)

✅ MediaPipe initialized successfully!
Version: 0.10.14


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
base_dir = '/content/drive/MyDrive/Linguisync3D'
print("Files in results:")
print(os.listdir(f"{base_dir}/results"))

Mounted at /content/drive
Files in results:
['test_audio.wav', 'sample_3d_landmarks.pt', 'linguisync3d_dubbed.mp4', 'hybrid_linguisync_dubbed.mp4', 'final_output_new.png', 'original_audio.wav', 'better_test_audio.wav', 'wav2lip_better_dubbed.mp4', 'joint_evaluation_final.png', 'final_grid_evaluation.png', 'final_wav2lip_dubbed.mp4', 'simple_linguisync_dubbed.mp4', 'joint_embedder.pth', 'smoothed_linguisync_dubbed.mp4', 'final_with_audio.mp4', 'test_audio_1.wav', 'test_audio_2.wav', 'test_audio_3.wav', 'test_audio_4.wav', 'test_audio_5.wav', 'dubbed_test_audio.mp4', 'dubbed_test_audio_1.mp4', 'dubbed_test_audio_2.mp4', 'dubbed_test_audio_3.mp4', 'dubbed_test_audio_4.mp4', 'dubbed_test_audio_5.mp4', 'comparison_table.csv', 'final_improved_dubbed.mp4', 'evaluation_metrics.csv']


In [ ]:
print("test_audio_1.wav exists?", os.path.exists(f"{base_dir}/results/test_audio_1.wav"))
print("test_audio_2.wav exists?", os.path.exists(f"{base_dir}/results/test_audio_2.wav"))

test_audio_1.wav exists? True
test_audio_2.wav exists? True


In [ ]:
import torch
import cv2
import numpy as np
import torchaudio
from transformers import Wav2Vec2Processor, Wav2Vec2Model
import mediapipe as mp

device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_dir = '/content/drive/MyDrive/Linguisync3D'

# MediaPipe
mp_face = mp.solutions.face_mesh.FaceMesh(
    static_image_mode=False, max_num_faces=1, refine_landmarks=True,
    min_detection_confidence=0.5, min_tracking_confidence=0.5
)

def extract_3d_landmarks(video_path, max_frames=50):
    cap = cv2.VideoCapture(video_path)
    landmarks_list = []
    idx = 0
    while cap.isOpened() and idx < max_frames:
        ret, frame = cap.read()
        if not ret: break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = mp_face.process(rgb)
        if results.multi_face_landmarks:
            lm = results.multi_face_landmarks[0]
            pts = np.array([[p.x, p.y, p.z] for p in lm.landmark])
        else:
            pts = np.zeros((478, 3))
        landmarks_list.append(pts)
        idx += 1
    cap.release()
    return torch.tensor(np.array(landmarks_list), dtype=torch.float32)

# Audio
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(device)
wav2vec.eval()

def get_audio_features(audio_path):
    waveform, sr = torchaudio.load(audio_path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(0, keepdim=True)
    if sr != 16000:
        waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
    inputs = processor(waveform.squeeze(0), sampling_rate=16000, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        return wav2vec(**inputs).last_hidden_state

class JointAudioVisualEmbedder(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.audio_proj = torch.nn.Linear(768, 512)
        self.video_proj = torch.nn.Linear(1434, 512)
    def forward(self, audio_feat, landmarks_3d):
        # Match lengths
        min_len = min(audio_feat.shape[1], landmarks_3d.shape[0])
        audio_feat = audio_feat[:, :min_len, :]
        landmarks_3d = landmarks_3d[:min_len, :, :].unsqueeze(0)
        B, T = audio_feat.shape[:2]
        audio_emb = self.audio_proj(audio_feat)
        video_flat = landmarks_3d.reshape(B, T, -1)
        video_emb = self.video_proj(video_flat)
        return audio_emb, video_emb

joint_embedder = JointAudioVisualEmbedder().to(device)

def joint_sync_loss(a, v):
    return torch.mean((a - v) ** 2)

print("✅ Fixed 3D Version Ready!")

# Run
source_video = f"{base_dir}/data/grid/s1_processed/s1_processed/bbaf2n.mpg"
target_audio = f"{base_dir}/results/test_audio_1.wav"

landmarks = extract_3d_landmarks(source_video, max_frames=100)
audio_feat = get_audio_features(target_audio)

with torch.no_grad():
    a_emb, v_emb = joint_embedder(audio_feat, landmarks)
    score = joint_sync_loss(a_emb, v_emb).item()

print(f"3D Joint Sync Score: {score:.4f}")

print("Generating video...")
%cd /content/Wav2Lip
!python inference.py --checkpoint_path checkpoints/Wav2Lip.pth \
    --face "{source_video}" --audio "{target_audio}" \
    --outfile "{base_dir}/results/3D_final_linguisync.mp4" --fps 25

print("✅ 3D Project Done!")

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Fixed 3D Version Ready!
3D Joint Sync Score: 0.1007
Generating video...
/content/Wav2Lip
Using cpu for inference.
Reading video frames...
Number of frames available for inference: 75
/content/Wav2Lip/audio.py:100: FutureWarning: Pass sr=16000, n_fft=800 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  return librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,
(80, 267)
Length of mel chunks: 80
  0% 0/1 [00:00<?, ?it/s]
  0% 0/5 [00:00<?, ?it/s]
 20% 1/5 [00:22<01:28, 22.02s/it]
 40% 2/5 [00:43<01:05, 21.99s/it]
 60% 3/5 [01:05<00:43, 21.66s/it]
 80% 4/5 [01:30<00:23, 23.22s/it]
100% 5/5 [01:46<00:00, 21.22s/it]
Load checkpoint from: checkpoints/Wav2Lip.pth
Model loaded
100% 1/1 [02:03<00:00, 123.90s/it]
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libd

In [ ]:
import cv2
import numpy as np

def add_smoothing(input_video, output_video, strength=3):
    cap = cv2.VideoCapture(input_video)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        frames.append(frame)
    cap.release()

    print(f"Smoothing {len(frames)} frames...")

    for i in range(len(frames)):
        start = max(0, i - strength)
        end = min(len(frames), i + strength + 1)
        smoothed = np.mean(frames[start:end], axis=0).astype(np.uint8)
        out.write(smoothed)

    out.release()
    print(f"✅ Smoothed video saved: {output_video}")

# Apply on your latest video
input_v = f"{base_dir}/results/3D_final_linguisync.mp4"
output_v = f"{base_dir}/results/smoothed_3D_linguisync.mp4"

add_smoothing(input_v, output_v)

Smoothing 80 frames...
✅ Smoothed video saved: /content/drive/MyDrive/Linguisync3D/results/smoothed_3D_linguisync.mp4


In [ ]:
target_audio = f"{base_dir}/results/test_audio_1.wav"  # change to 2,3... for different tests
output_video = f"{base_dir}/results/improved_lip_sync.mp4"

print("Generating improved lip sync...")

%cd /content/Wav2Lip
!python inference.py --checkpoint_path checkpoints/Wav2Lip.pth \
    --face "{base_dir}/data/grid/s1_processed/s1_processed/bbaf2n.mpg" \
    --audio "{target_audio}" \
    --outfile "{output_video}" --fps 25

print(f"✅ Improved lip sync saved: {output_video}")

Generating improved lip sync...
/content/Wav2Lip
Using cpu for inference.
Reading video frames...
Number of frames available for inference: 75
/content/Wav2Lip/audio.py:100: FutureWarning: Pass sr=16000, n_fft=800 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  return librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,
(80, 267)
Length of mel chunks: 80
  0% 0/1 [00:00<?, ?it/s]
  0% 0/5 [00:00<?, ?it/s]
 20% 1/5 [00:23<01:32, 23.21s/it]
 40% 2/5 [00:44<01:06, 22.23s/it]
 60% 3/5 [01:07<00:44, 22.44s/it]
 80% 4/5 [01:32<00:23, 23.39s/it]
100% 5/5 [01:47<00:00, 21.42s/it]
Load checkpoint from: checkpoints/Wav2Lip.pth
Model loaded
100% 1/1 [02:05<00:00, 125.45s/it]
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr